In [1]:
import os
import pandas as pd

# Odczyt datasetu

In [2]:
df = pd.read_csv(os.path.join("../data", "domy.csv"))

# Usuwanie kolumn

# Zamiana kilku danych tekstowych na numeryczne

In [3]:
del_cols = ["Id", "Street", "Utilities", "Condition2", "PoolQC", "PoolArea", "LowQualFinSF", "3SsnPorch"]
df = df.copy().drop(del_cols, axis=1)

In [4]:
df = df.copy()
df["LotFrontage"] = pd.to_numeric(df["LotFrontage"], errors="coerce")
df["MasVnrArea"] = pd.to_numeric(df["MasVnrArea"], errors="coerce")
df["GarageYrBlt"] = pd.to_numeric(df["GarageYrBlt"], errors="coerce")

# Usuwanie outlierów

In [5]:
out_cols = ["LotArea", "GrLivArea", "Fireplaces", "HalfBath", "GarageCars"]
ranges = [100000, 5000, 2, 1, 3]

df = df.copy()
for i in range(len(out_cols)):
    df = df[df[out_cols[i]] <= ranges[i]]

# Podział danych na zbiór treningowy i testowy

In [6]:
from sklearn.model_selection import train_test_split

X = df.drop("SalePrice", axis=1)
y = df["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Trening

In [7]:
from Regression.training_reporter import TrainingReporter
from Regression.training_model import CustomXGBRegressorModel

model = CustomXGBRegressorModel()
reporter = TrainingReporter(model, X_train, X_test, y_train, y_test)

reporter.train()

Start training...
Training finished!
---------------------------------------------------


# Kroswalidacja

In [8]:
reporter.run_cross_validation(cv=10)

Start cross validation...
Fold 0: RMSE 30630.741159821777
Fold 1: RMSE 29173.438878541558
Fold 2: RMSE 22934.39547927959
Fold 3: RMSE 24078.78668039567
Fold 4: RMSE 21857.382825946934
Fold 5: RMSE 47943.34523163773
Fold 6: RMSE 29916.342022379675
Fold 7: RMSE 37640.75918469233
Fold 8: RMSE 27471.146171938293
Fold 9: RMSE 21192.2113994741
RMSE for cross validation: 29283.854903410764 +- 7820.244420475953
Cross validation finished!
---------------------------------------------------


# Grid search

In [9]:
reporter.run_grid_search(cv=5)

Start grid search...
Grid finished!
Best params: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__min_child_weight': 3, 'model__n_estimators': 1000}


# Kroswalidacja po dostosowaniu hiperparametrów

In [10]:
model = CustomXGBRegressorModel(n_estimators=1000, learning_rate=0.05, max_depth=3, min_child_weight=3)
reporter = TrainingReporter(model, X_train, X_test, y_train, y_test)
reporter.run_cross_validation(cv=10)

Start cross validation...
Fold 0: RMSE 27260.02083638235
Fold 1: RMSE 26508.752969538193
Fold 2: RMSE 24189.43240342774
Fold 3: RMSE 21941.06797765323
Fold 4: RMSE 20727.825549246598
Fold 5: RMSE 50314.53324835678
Fold 6: RMSE 26592.35258490681
Fold 7: RMSE 33015.17711598713
Fold 8: RMSE 28460.13464479745
Fold 9: RMSE 22261.595270779675
RMSE for cross validation: 28127.089260107594 +- 8154.375227688505
Cross validation finished!
---------------------------------------------------
